# c14.co.il — passive recon

Runs the passive-only recon tools from `pentest-c14/tools/` (see
`scope.md` in the repo — this engagement starts passive-only, same as
`pentest-milatova/` did, until scope.md is updated to authorize active
testing for a specific finding/CVE).

1. **wp-json + disclosure paths** — `wp-json` root/users/plugins,
   `robots.txt`, `sitemap.xml`, `readme.html`, header fingerprinting.
2. **Login/auth hardening** — `xmlrpc.php`, `wp-login.php` header
   fingerprinting, `?author=N` redirect enumeration (no credentials
   ever submitted).
3. **Deep recon** — sensitive-path exposure checks, plugin version
   fingerprinting, `wp-json` subroute anonymous-access checks.
4. **WAF/bot-protection summary** — every request across all sections
   is checked for known WAF block-page signatures (Cloudflare, Sucuri,
   Imperva/Incapsula, Akamai, AWS WAF). If most/all requests come back
   blocked — including harmless paths like `/robots.txt` — that's a
   strong signal the block is on User-Agent/source-IP (e.g. Colab's
   Google Cloud IP range gets blocklisted by many WAFs as automation
   traffic), not real per-endpoint access control.

If this site isn't WordPress, the WordPress-specific checks will
mostly 404 — that's expected and itself useful signal about the stack.

This notebook is self-contained (stdlib only, no `pip install` needed)
so it runs in a fresh Colab runtime as-is.

### If every request comes back blocked (WAF)

If section 4 reports most/all requests blocked, that's most likely
Cloudflare (or similar) rejecting Colab's IP range wholesale, not a
per-endpoint decision. As the site owner, you can configure a Cloudflare
custom rule yourself — "skip security if header `X-Pentest-Token`
equals `<a secret only you generate>`" — and this notebook will send
that header if you set it up:

1. In Colab's left sidebar, click the **key icon** (Secrets), add a
   secret named `PENTEST_BYPASS_TOKEN_C14` with your chosen value, and
   grant this notebook access.
2. Create the matching Cloudflare rule with that same value.

**Do not** type the secret directly into the `BYPASS_HEADER_VALUE_MANUAL`
field below if you intend to save/commit this notebook afterward — use
Colab Secrets instead, which never gets written into the notebook file.
This is intentionally NOT a User-Agent-based allowlist: this notebook's
UA string is public (it's in this GitHub repo), so a UA-based Cloudflare
rule could be trivially spoofed by anyone who read the code.

**Only run this against a target you are authorized to test.**


In [ ]:
TARGET = "https://c14.co.il"  #@param {type:"string"}
DELAY = 1.5  #@param {type:"number"}
TIMEOUT = 10.0  #@param {type:"number"}

# Optional WAF-bypass header, for a Cloudflare custom rule YOU configure
# yourself (e.g. "skip security if header X-Pentest-Token equals
# <secret>"). Do NOT put the secret value directly in the field below if
# you ever save/commit this notebook -- use Colab's built-in Secrets
# instead (key icon in the left sidebar: add a secret named
# PENTEST_BYPASS_TOKEN_C14, grant this notebook access). The manual field is
# only a fallback for quick one-off runs you don't save afterward.
BYPASS_HEADER_NAME = "X-Pentest-Token"  #@param {type:"string"}
BYPASS_HEADER_VALUE_MANUAL = ""  #@param {type:"string"}

import json
import re
import time
import urllib.error
import urllib.request

USER_AGENT = "pentest-c14-colab-recon/1.0 (+authorized-assessment)"

BYPASS_HEADER_VALUE = BYPASS_HEADER_VALUE_MANUAL
try:
    from google.colab import userdata
    _secret = userdata.get("PENTEST_BYPASS_TOKEN_C14")
    if _secret:
        BYPASS_HEADER_VALUE = _secret
except Exception:
    pass  # not in Colab, or no secret configured -- fall back to manual field above

if BYPASS_HEADER_NAME and BYPASS_HEADER_VALUE:
    print(f"WAF-bypass header configured: {BYPASS_HEADER_NAME} (value hidden)")

def fetch(url, timeout=TIMEOUT, method="GET", follow_redirects=True):
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **kw):
            return None
    opener = (urllib.request.build_opener(NoRedirect)
              if not follow_redirects else urllib.request.build_opener())
    req_headers = {"User-Agent": USER_AGENT}
    if BYPASS_HEADER_NAME and BYPASS_HEADER_VALUE:
        req_headers[BYPASS_HEADER_NAME] = BYPASS_HEADER_VALUE
    req = urllib.request.Request(url, headers=req_headers, method=method)
    try:
        with opener.open(req, timeout=timeout) as resp:
            return resp.status, dict(resp.headers), resp.read()
    except urllib.error.HTTPError as e:
        return e.code, dict(e.headers or {}), e.read()
    except urllib.error.URLError as e:
        return None, {}, str(e).encode()

base = TARGET.rstrip("/")
print(f"Target: {base}")

# A blanket 401/403/429 across nearly every path (including harmless
# ones) usually means a WAF/bot-protection layer is blocking on
# User-Agent or source IP/ASN (e.g. Colab's Google Cloud IP range),
# not real per-path access control.
WAF_SIGNATURES = {
    "cloudflare": ["cloudflare", "attention required", "cf-ray", "cf-mitigated"],
    "sucuri": ["sucuri website firewall", "sucuri/cloudproxy"],
    "imperva/incapsula": ["incapsula", "imperva", "_incap_ses"],
    "akamai": ["akamai", "ak_bmsc"],
    "aws-waf": ["aws waf", "x-amzn-requestid", "awselb"],
    "generic-block-page": ["access denied", "request blocked", "bot detected"],
}

def waf_hint(status, headers, body):
    if status not in (401, 403, 429):
        return None
    blob = (body[:2000].decode("utf-8", errors="replace") + " "
            + " ".join(f"{k}:{v}" for k, v in headers.items())).lower()
    for vendor, needles in WAF_SIGNATURES.items():
        if any(n in blob for n in needles):
            return vendor
    return "unknown-vendor"

waf_request_count = 0
waf_blocked_count = 0
waf_vendors_seen = set()

def track_waf(status, headers, body):
    global waf_request_count, waf_blocked_count
    waf_request_count += 1
    hint = waf_hint(status, headers, body)
    if hint:
        waf_blocked_count += 1
        waf_vendors_seen.add(hint)
    return hint


## 1. wp-json + disclosure paths


In [ ]:
ROUTES = [
    ("root", "/wp-json/", "json"),
    ("users", "/wp-json/wp/v2/users", "json"),
    ("plugins", "/wp-json/wp/v2/plugins", "json"),
    ("robots", "/robots.txt", "text"),
    ("sitemap", "/sitemap.xml", "text"),
    ("readme", "/readme.html", "text"),
]

def fingerprint_headers(headers):
    interesting = ["Server", "X-Powered-By", "Link", "X-Generator"]
    hits = {k: v for k, v in headers.items() if k in interesting}
    if hits:
        print("  headers:", ", ".join(f"{k}={v}" for k, v in hits.items()))

print("== wp-json + disclosure paths ==")
discovered_author_ids = []
for name, path, kind in ROUTES:
    url = base + path
    status, headers, body = fetch(url)
    print(f"[{status}] GET {url}")
    fingerprint_headers(headers)
    hint = track_waf(status, headers, body)
    if hint:
        print(f"  ** likely WAF/bot-protection block ({hint})")
    text = body.decode("utf-8", errors="replace") if body else ""

    if status == 200 and name == "root" and kind == "json":
        try:
            namespaces = json.loads(text).get("namespaces", [])
            print(f"  {len(namespaces)} REST namespace(s): {namespaces}")
        except (json.JSONDecodeError, AttributeError):
            pass
    elif status == 200 and name == "users":
        try:
            for u in json.loads(text):
                print(f"  user: id={u.get('id')} slug={u.get('slug')} name={u.get('name')}")
                if u.get("id") is not None:
                    discovered_author_ids.append(u["id"])
        except (json.JSONDecodeError, TypeError):
            pass
    elif status == 200 and name == "readme":
        m = re.search(r"Version\s+([\d.]+)", text)
        if m:
            print(f"  WordPress version disclosed: {m.group(1)}")
    time.sleep(DELAY)

print(f"\nDiscovered author IDs: {discovered_author_ids or '(none -- section 2 will fall back to probing 1-10)'}")


## 2. Login/auth hardening (passive, no credentials submitted)


In [ ]:
def security_plugin_fingerprint(headers):
    hits = []
    blob = " ".join(f"{k}:{v}" for k, v in headers.items()).lower()
    signals = {
        "wordfence": ["wordfence", "wfwaf", "wf-"],
        "sucuri": ["sucuri", "x-sucuri"],
        "cloudflare": ["cf-ray", "cloudflare"],
        "generic-waf-cookie": ["wfvt_", "wf_"],
    }
    for plugin, needles in signals.items():
        if any(n in blob for n in needles):
            hits.append(plugin)
    return hits

print("== Login/auth hardening ==")

url = f"{base}/xmlrpc.php"
status, headers, body = fetch(url)
print(f"[{status}] GET {url}")
text = body.decode("utf-8", errors="replace")
if status == 200 and "XML-RPC server accepts POST requests only" in text:
    print("  xmlrpc.php is ENABLED and reachable (potential brute-force/pingback-amplification vector)")
elif status in (403, 404, None):
    print("  xmlrpc.php appears blocked/disabled (good)")
else:
    print(f"  xmlrpc.php returned unexpected status {status}, inspect manually")
fp = security_plugin_fingerprint(headers)
if fp:
    print(f"  security-plugin signals: {fp}")
hint = track_waf(status, headers, body)
if hint:
    print(f"  ** likely WAF/bot-protection block ({hint})")
time.sleep(DELAY)

url = f"{base}/wp-login.php"
status, headers, body = fetch(url)
print(f"[{status}] GET {url}")
fp = security_plugin_fingerprint(headers)
if fp:
    print(f"  security-plugin signals: {fp}")
else:
    print("  no WAF/security-plugin signal detected in headers (inconclusive)")
hint = track_waf(status, headers, body)
if hint:
    print(f"  ** likely WAF/bot-protection block ({hint})")
time.sleep(DELAY)

author_ids = discovered_author_ids if discovered_author_ids else list(range(1, 11))
print(f"\nAuthor-ID redirect enumeration (no-follow), IDs: {author_ids}")
for aid in author_ids:
    url = f"{base}/?author={aid}"
    status, headers, body = fetch(url, follow_redirects=False)
    location = headers.get("Location", "")
    print(f"  [{status}] ?author={aid} -> Location: {location}")
    track_waf(status, headers, body)
    time.sleep(DELAY)


## 3. Deep recon: sensitive paths + plugin fingerprinting


In [ ]:
SENSITIVE_PATHS = [
    "/.git/HEAD",
    "/.git/config",
    "/.env",
    "/wp-config.php.bak",
    "/wp-config.php~",
    "/wp-config.php.save",
    "/wp-config.php.old",
    "/wp-content/debug.log",
    "/.well-known/security.txt",
    "/security.txt",
    "/backup.sql",
    "/database.sql",
    "/wp-content/uploads/",  # directory listing check only
    "/.svn/entries",
    "/.DS_Store",
    "/composer.json",
    "/composer.lock",
    "/package.json",
    "/vendor/composer/installed.json",
    "/license.txt",
    "/error_log",
    "/wp-content/error_log",
    "/phpinfo.php",
    "/info.php",
    "/server-status",  # Apache mod_status
    "/wp-content/uploads/wp-config.php",  # accidental relocation
]

# Plugins to fingerprint via their public readme.txt (standard WP
# convention: "Stable tag: X.Y.Z"). Namespace-confirmed (woocommerce,
# elementor via wp/v2/users meta) plus common high-impact plugins worth
# ruling in/out.
PLUGIN_README_PATHS = {
    "woocommerce": "/wp-content/plugins/woocommerce/readme.txt",
    "elementor": "/wp-content/plugins/elementor/readme.txt",
    "jetpack": "/wp-content/plugins/jetpack/readme.txt",
    "wordfence": "/wp-content/plugins/wordfence/readme.txt",
    "yoast-seo": "/wp-content/plugins/wordpress-seo/readme.txt",
    "contact-form-7": "/wp-content/plugins/contact-form-7/readme.txt",
    "updraftplus": "/wp-content/plugins/updraftplus/readme.txt",
    "wpforms": "/wp-content/plugins/wpforms-lite/readme.txt",
    "all-in-one-seo": "/wp-content/plugins/all-in-one-seo-pack/readme.txt",
    "duplicator": "/wp-content/plugins/duplicator/readme.txt",
    "wp-fastest-cache": "/wp-content/plugins/wp-fastest-cache/readme.txt",
    "litespeed-cache": "/wp-content/plugins/litespeed-cache/readme.txt",
    "wp-super-cache": "/wp-content/plugins/wp-super-cache/readme.txt",
    "acf": "/wp-content/plugins/advanced-custom-fields/readme.txt",
    "really-simple-ssl": "/wp-content/plugins/really-simple-ssl/readme.txt",
    "loginizer": "/wp-content/plugins/loginizer/readme.txt",
}

VERSION_RE = re.compile(r"Stable tag:\s*([\d.]+)", re.IGNORECASE)

# Additional wp-json subroutes worth checking for anonymous access --
# still GET-only. A 200 here isn't necessarily a bug (some are meant to
# be public), but several (settings, block-renderer) should require
# auth, so it's worth recording status codes for the report.
WP_JSON_SUBROUTES = [
    "/wp-json/wp/v2/media",
    "/wp-json/wp/v2/comments",
    "/wp-json/wp/v2/categories",
    "/wp-json/wp/v2/tags",
    "/wp-json/wp/v2/types",
    "/wp-json/wp/v2/settings",  # should require auth
    "/wp-json/wp/v2/block-renderer",  # should require auth
    "/wp-json/oembed/1.0/embed",
]


VERSION_RE = re.compile(r"Stable tag:\s*([\d.]+)", re.IGNORECASE)

print("== Sensitive path exposure checks ==")
for path in SENSITIVE_PATHS:
    url = base + path
    status, headers, resp_body = fetch(url)
    flag = " <-- EXPOSED, investigate" if status == 200 else ""
    hint = track_waf(status, headers, resp_body)
    if hint:
        flag += f" [WAF/bot-protection block? {hint}]"
    print(f"[{status}] GET {url}{flag}")
    time.sleep(DELAY)

print("\n== Plugin version fingerprinting (readme.txt Stable tag) ==")
for name, path in PLUGIN_README_PATHS.items():
    url = base + path
    status, headers, resp_body = fetch(url)
    version = None
    if status == 200:
        m = VERSION_RE.search(resp_body.decode("utf-8", errors="replace"))
        version = m.group(1) if m else None
    track_waf(status, headers, resp_body)
    print(f"[{status}] {name}: {url}" + (f" -> version {version}" if version else ""))
    time.sleep(DELAY)

print("\n== wp-json root: allowed HTTP methods (OPTIONS) ==")
url = base + "/wp-json/"
status, headers, resp_body = fetch(url, method="OPTIONS")
track_waf(status, headers, resp_body)
print(f"[{status}] OPTIONS {url} -> Allow: {headers.get('Allow', 'n/a')}")
time.sleep(DELAY)

print("\n== wp-json subroute anonymous-access checks ==")
for path in WP_JSON_SUBROUTES:
    url = base + path
    status, headers, resp_body = fetch(url)
    flag = ""
    if status == 200 and path.rsplit("/", 1)[-1] in ("settings", "block-renderer"):
        flag = " <-- should require auth, investigate"
    track_waf(status, headers, resp_body)
    print(f"[{status}] GET {url}{flag}")
    time.sleep(DELAY)


## 4. Overall WAF/bot-protection summary


In [ ]:
print(f"{waf_blocked_count}/{waf_request_count} requests across all sections looked like "
      f"WAF/bot-protection blocks.")
if waf_vendors_seen:
    print(f"Possible vendor(s): {', '.join(sorted(waf_vendors_seen))}")

if waf_request_count and waf_blocked_count >= waf_request_count * 0.5:
    print("\nThis pattern -- blocked on harmless paths too (e.g. /robots.txt), not just "
          "sensitive ones -- usually means the block is on User-Agent or source IP/ASN "
          "(Colab runs on Google Cloud IPs, which many WAFs blocklist wholesale as "
          "known automation/scraping sources), not real per-path access control.\n\n"
          "To confirm: try re-running from a different network (e.g. your own machine "
          "instead of Colab), or note this as a passive finding as-is -- 'site has "
          "active bot protection that blocks non-browser/datacenter traffic' is itself "
          "a legitimate (if unremarkable) recon result, not a failed scan.")
elif waf_blocked_count == 0:
    print("\nNo WAF/bot-protection signals detected across any section.")
